In [10]:
import torch
import networkx as nx
from torch_geometric.data import Dataset, Data
from torch_geometric.utils import from_networkx

In [11]:
class GPDataset(Dataset):
    def __init__(self, task='Connected', difficulty='easy', root=None, transform=None, pre_transform=None):
        self.task_name = task
        self.difficulty = difficulty
        super().__init__(root, transform, pre_transform)
        
        dataset_loc =  'dataset'
        task = globals()[task+'_Task'](dataset_loc)
        task.load_dataset(difficulty)
        graph_problems = task.problem_set
        
        self.data_list = []
        for gp in graph_problems:
            if gp['exact_answer'] == None:
                continue
            if len(gp['graph']) == 2:
                gp['graph'] = nx.disjoint_union(gp['graph'][0], gp['graph'][1])
            
            node_dict = {j:i for i, j in enumerate(gp['graph'].nodes())}
            
            # Convert NetworkX graph to PyG Data object
            data = from_networkx(gp['graph'])
            
            # Add node features
            data.x = torch.ones(data.num_nodes, 1)
            
            # Add label
            data.y = torch.tensor([gp['exact_answer']], dtype=torch.long)
            
            # Handle source/target nodes
            if 'node1' in gp:
                gp['source'] = gp['node1']
            if 'node2' in gp:
                gp['target'] = gp['node2']
            
            if 'source' in gp:
                gp['source'] = node_dict[gp['source']]
                source_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
                source_mask[gp['source']] = True
                data.source = source_mask
            
            if 'target' in gp:
                gp['target'] = node_dict[gp['target']]
                target_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
                target_mask[gp['target']] = True
                data.target = target_mask
            
            self.data_list.append(data)
    
    def len(self):
        return len(self.data_list)
    
    def get(self, idx):
        return self.data_list[idx]

In [12]:
# Import task classes
from tasks import *

# Load example graphs from different tasks
# You can change the task, mode, and difficulty parameters as needed
task_name = 'Connected'  # Options: Connected, Diameter, Distance, GED, MCP, MCS, MIS, MVC, Neighbor, TSP
difficulty = 'hard'  # Options: 'easy', 'medium', 'hard'

# Create dataset
dataset = GPDataset(task=task_name, difficulty=difficulty)
print(f"Loaded {len(dataset)} graphs")
print(f"\nExample graph (index 0):")
example = dataset[0]
print(f"  Number of nodes: {example.num_nodes}")
print(f"  Number of edges: {example.num_edges}")
print(f"  Node features shape: {example.x.shape}")
print(f"  Label: {example.y.item()}")
if hasattr(example, 'source'):
    print(f"  Has source node: {example.source.sum().item()} node(s)")
if hasattr(example, 'target'):
    print(f"  Has target node: {example.target.sum().item()} node(s)")

Loaded 500 graphs

Example graph (index 0):
  Number of nodes: 15
  Number of edges: 30
  Node features shape: torch.Size([15, 1])
  Label: 1
